# 比特频谱扫描

一次调用只执行给定频率范围和步进的一次扫描。实验结果自动写入 `output/experiments/`，Web 控制台负责查看结果。

In [9]:
from uuid import uuid4

from sqvm.calibration import (
    apply_calibration_candidates_to_current_configuration,
    run_spectroscopy,
)

print('SQVM 用户接口导入成功')

SQVM 用户接口导入成功


## 扫描参数

每个对象填写 `(起始频率, 终止频率)`，单位为 GHz。一个对象执行单比特扫描，两个对象自动并行；并行时两个范围必须按公共步进产生相同数量的数据点。脉冲和设备参数使用 API 默认值。

In [ ]:
FREQUENCY_RANGES_GHZ = {
    'Q1': (5.00, 5.40),
    'Q2': (5.10, 5.50),
}
FREQUENCY_STEP_GHZ = 0.02
OPERATION_ID = str(uuid4())
RUN_EXPERIMENT = True
UPDATE_PARAMETERS = False

In [11]:
def report_progress(event):
    action = '开始' if event['event'] == 'circuit_started' else '完成'
    current = event['completed'] + 1 if event['event'] == 'circuit_started' else event['completed']
    print(f"{action} {current}/{event['total']}: {event['circuit_id']}")

if RUN_EXPERIMENT:
    result = run_spectroscopy(
        FREQUENCY_RANGES_GHZ,
        frequency_step_GHz=FREQUENCY_STEP_GHZ,
        operation_id=OPERATION_ID,
        progress_callback=report_progress,
    )
    print(f'运行 ID: {result.run_id}')
    print(f'结果目录: {result.root}')
    for target, data in result.data.items():
        print(f'{target} frequency_GHz: {list(data["frequency_GHz"])}')
        print(f'{target} P1: {list(data["P1"])}')
        print(f'{target} peak: {result.analysis.peaks[target]}')
        print(f'{target} candidate: {result.candidates[target]}')
else:
    print('参数已设置。将 RUN_EXPERIMENT 改为 True 后重新运行本单元格。')

开始 1/6: sp_4a03afd6891055eb40df520e0a081286
完成 1/6: sp_4a03afd6891055eb40df520e0a081286
开始 2/6: sp_4d85b8a850e9fb7a199bb2ce2ae7b0a2
完成 2/6: sp_4d85b8a850e9fb7a199bb2ce2ae7b0a2
开始 3/6: sp_03dcc01f88e8cc0652dd78e71292bc95
完成 3/6: sp_03dcc01f88e8cc0652dd78e71292bc95
开始 4/6: sp_d49c2214290a581932d4e044949f72d0
完成 4/6: sp_d49c2214290a581932d4e044949f72d0
开始 5/6: sp_80a7f58e5d3b101eba274540c8498f6c
完成 5/6: sp_80a7f58e5d3b101eba274540c8498f6c
开始 6/6: sp_ae29b42bbc8225b1e03f9629bf36702e
完成 6/6: sp_ae29b42bbc8225b1e03f9629bf36702e
运行 ID: 0eca61f1-6038-4e64-8d34-2004e4714aef
结果目录: D:\Codex\SQC_simulation\output\experiments\qubit_spectroscopy_0eca61f160384e648d342004e4714aef
Q2 frequency_GHz: [5.3, 5.32, 5.34, 5.36, 5.38, 5.4]
Q2 P1: [0.24445924830671129, 0.3891578253963526, 0.40732003076041073, 0.28257145154717217, 0.12284054128631447, 0.02805058979603796]
Q2 peak: SpectroscopyPeak(qagent='Q2', valid=True, discrete_frequency_GHz=5.34, estimated_frequency_GHz=5.332541754342107, peak_population=0.

## 候选校准值

本次扫描的有效峰值会形成候选频率，但不会自动修改配置。检查候选、对比度和门限后，将 `UPDATE_PARAMETERS` 改为 `True`，再运行下方单元格。

In [12]:
if UPDATE_PARAMETERS:
    if not RUN_EXPERIMENT or 'result' not in globals():
        raise RuntimeError('请先运行实验并检查候选校准值')
    confirmation = f'APPLY CALIBRATION CANDIDATES {result.run_id}'
    update = apply_calibration_candidates_to_current_configuration(
        result,
        confirmation_phrase=confirmation,
    )
    print(f'已更新当前配置: {update.device_id} r{update.current_revision}')
    print(f'更新目标: {update.targets}')
    print(f'已更新参数: {dict(update.applied_values)}')
else:
    print('候选值尚未写入当前配置；确认后将 UPDATE_PARAMETERS 改为 True。')

候选值尚未写入当前配置；确认后将 UPDATE_PARAMETERS 改为 True。


## 查看结果

启动 `start_calibration_web.cmd` 后访问 [http://127.0.0.1:8765/#/experiments](http://127.0.0.1:8765/#/experiments)。